> **Versión resuelta (para docentes).** No compartir con los alumnos.

# Unidad 6 · Bibliotecas — Sherlock Forense

Caso **ficticio**: robaron una obra de la bóveda del museo. El sistema de sensores registró
cuántas **pisadas** dejó cada sospechoso en cada sala. Esa es la evidencia.
- `huellas.csv` → `sospechoso, sala, pisadas`
- `indicios.csv` → `sospechoso, cabellos`

**Para empezar:** descargá los CSV desde la página de la clase y subilos a Colab (panel 📁).
El registro es un desastre: hay **decenas de sospechosos** (muchos sin sentido en el caso: hasta Godzilla, un pingüino y Keyboard Cat) y un montón de salas. La `Boveda` es la escena. Los que importan: **Moran, Adler, Hudson, Lestrade, Gregson, Moriarty, Watson, Baynes**.
Ojo: el que más pisadas dejó **en total** no es necesariamente el culpable (un guardia patrulla todo el museo); mirá quién se **concentró en la bóveda**.
Cada consigna está en **estilo parcial**: Sherlock te pone al frente del análisis y te pide algo.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv('huellas.csv')
df.head()

## Parte 1 · pandas (seguir las huellas)

**1.** Sherlock te pone al frente del análisis y te pasa `huellas.csv`. Te pide
`total_pisadas_en_sala(archivo, sala)` que **devuelva** el total de pisadas registradas en esa
sala (sumando a todos los sospechosos).

In [ ]:
def total_pisadas_en_sala(archivo, sala):
    df = pd.read_csv(archivo)
    de_la_sala = df[df['sala'] == sala]
    return de_la_sala['pisadas'].sum()

print(total_pisadas_en_sala('huellas.csv', 'Boveda'))  # 117

**2.** Para saber quién estuvo en el lugar del crimen, Sherlock te pide el desglose por
sospechoso de una sala: `pisadas_por_sospechoso(archivo, sala)` → `{sospechoso: pisadas}`.

In [ ]:
def pisadas_por_sospechoso(archivo, sala):
    df = pd.read_csv(archivo)
    de_la_sala = df[df['sala'] == sala]
    return de_la_sala.groupby('sospechoso')['pisadas'].sum().to_dict()

print(pisadas_por_sospechoso('huellas.csv', 'Boveda'))

**3.** Sherlock sospecha de quien más se movió por toda la escena. Te pide
`sospechoso_mas_presente(archivo)` → el sospechoso con más pisadas sumando todas las salas.

In [ ]:
def sospechoso_mas_presente(archivo):
    df = pd.read_csv(archivo)
    return df.groupby('sospechoso')['pisadas'].sum().idxmax()

print(sospechoso_mas_presente('huellas.csv'))  # El Gato del Museo: se movió por todo el museo, no es el culpable

## Parte 2 · numpy (las mediciones del perito)

**4.** El perito midió la **profundidad** (en mm) de varias pisadas y Sherlock te pide el valor
típico. `promedio(valores)` → el promedio redondeado a 2 decimales.

In [ ]:
def promedio(valores):
    return round(float(np.array(valores).mean()), 2)

print(promedio([22, 24, 21, 23, 25, 60, 22, 23]))  # 27.5

**5.** Entre las profundidades hay una que no encaja: la pisada del culpable, la que **más se
aleja del promedio**. Sherlock te pide `medida_mas_rara(valores)` → la posición (índice) de esa
medición. Ej: `medida_mas_rara([22, 24, 21, 23, 25, 60, 22, 23])` → `5`.

In [ ]:
def medida_mas_rara(valores):
    a = np.array(valores)
    distancias = np.abs(a - a.mean())
    return int(distancias.argmax())

print(medida_mas_rara([22, 24, 21, 23, 25, 60, 22, 23]))  # 5

## Parte 3 · matplotlib

**6. Barras de sospechosos.** Graficá con `plt.bar` el total de pisadas por sospechoso.

In [ ]:
totales = df.groupby('sospechoso')['pisadas'].sum()

fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(totales.index, totales.values)
ax.set_title('Pisadas totales por sospechoso')
ax.set_xlabel('Sospechoso')
ax.set_ylabel('Pisadas')
ax.tick_params(axis='x', rotation=90)
plt.show()

**7.** Marcá en un gráfico de las profundidades la medición más rara (`plt.scatter` en rojo).

In [ ]:
profundidades = [22, 24, 21, 23, 25, 60, 22, 23]
rara = medida_mas_rara(profundidades)

fig, ax = plt.subplots()
ax.plot(profundidades, marker='o', label='Profundidad (mm)')
ax.scatter(rara, profundidades[rara], color='red', label='Medición más rara')
ax.set_title('Profundidad de las pisadas')
ax.set_xlabel('Medición')
ax.set_ylabel('Profundidad (mm)')
ax.legend()
ax.grid()
plt.show()

## Integrador · estilo parcial

**Base:** archivos, pandas, numpy y matplotlib. **Siempre graficar.** La escena del crimen es la `Boveda`.

**Inciso A.** Escribí `analizar_y_graficar_sospechas(ruta_archivo)` que: (1) calcule con **NumPy**
el promedio de pisadas por registro de todo el archivo; (2) calcule, para cada sospechoso, su
**índice de concentración en la escena** `indice = (pisadas_en_la_Boveda / pisadas_totales_del_sospechoso) * 100`;
y (3) grafique un **gráfico de barras** con los sospechosos en el eje X y su índice (%) en el eje Y.
El pico es el principal sospechoso.

In [ ]:
def analizar_y_graficar_sospechas(ruta_archivo):
    df = pd.read_csv(ruta_archivo)

    # (1) promedio de pisadas por registro, con NumPy
    promedio_registro = np.array(df['pisadas']).mean()
    print('Promedio de pisadas por registro:', round(float(promedio_registro), 2))

    # (2) índice de concentración en la escena para cada sospechoso
    sospechosos = list(df['sospechoso'].unique())
    indices = []
    for sospechoso in sospechosos:
        del_sospechoso = df[df['sospechoso'] == sospechoso]
        total = del_sospechoso['pisadas'].sum()
        en_boveda = del_sospechoso[del_sospechoso['sala'] == 'Boveda']['pisadas'].sum()
        indices.append(en_boveda / total * 100)

    # (3) gráfico de barras: el pico es el principal sospechoso
    fig, ax = plt.subplots(figsize=(14, 5))
    ax.bar(sospechosos, indices)
    ax.set_title('Índice de concentración en la Bóveda')
    ax.set_xlabel('Sospechoso')
    ax.set_ylabel('Índice (%)')
    ax.tick_params(axis='x', rotation=90)
    plt.show()

    return dict(zip(sospechosos, indices))

indices = analizar_y_graficar_sospechas('huellas.csv')
print(max(indices, key=indices.get), round(float(max(indices.values())), 1))  # Moran 50.0

**Inciso B.** Escribí `exportar_sospechosos_sala(ruta_archivo, sala)` que, con **Pandas**:
(1) filtre solo las filas de esa sala; (2) las ordene de mayor a menor por `pisadas`; y
(3) exporte el resultado a un CSV `sospechosos_sala.csv`.

In [ ]:
def exportar_sospechosos_sala(ruta_archivo, sala):
    df = pd.read_csv(ruta_archivo)
    de_la_sala = df[df['sala'] == sala]
    ordenado = de_la_sala.sort_values(by=['pisadas'], ascending=False)
    ordenado.to_csv('sospechosos_sala.csv', index=False)

exportar_sospechosos_sala('huellas.csv', 'Boveda')
pd.read_csv('sospechosos_sala.csv').head()

## Caso cerrado · ¿Quién fue?

Hora de acusar: Sherlock te pide el veredicto. `identificar_culpable(archivo, sala)` → el
sospechoso con más pisadas en esa sala (la escena es `Boveda`).

In [ ]:
def identificar_culpable(archivo, sala):
    df = pd.read_csv(archivo)
    de_la_sala = df[df['sala'] == sala]
    return de_la_sala.groupby('sospechoso')['pisadas'].sum().idxmax()

print(identificar_culpable('huellas.csv', 'Boveda'))  # Moran

**Acusación final: fue Moran.**

- **Presencia en la bóveda:** dejó 22 de las 117 pisadas de la `Boveda`, más que cualquier otro sospechoso (el segundo, Moriarty, dejó 9).
- **Índice de concentración:** 50%. La mitad de todas sus pisadas fueron en la escena del crimen; nadie se le acerca (Moriarty, 26.5%).
- **Total de pisadas:** el que más pisadas dejó en todo el museo es El Gato del Museo (187), pero se movió por todas las salas: su total alto no lo pone en la bóveda.
- **Cabellos:** dejó 25 cabellos en la escena (`indicios.csv`).
- **La pisada más rara:** la medición de 60 mm (posición 5) se despega del resto, que ronda los 22–25 mm.

## Peritaje de código

Dos peritos entregaron estas funciones y dan mal. Encontrá y corregí los errores.

**a.** Debería devolver la lista de sospechosos que dejaron más de 10 cabellos (`indicios.csv`):

In [ ]:
def sospechosos_con_cabellos(archivo):
    df = pd.read_csv(archivo)
    mayores = df[df['cabellos'] > 10]          # comparar contra el número 10, no contra el texto '10'
    return list(mayores['sospechoso'])         # devolver (no imprimir) solo los nombres

print(sospechosos_con_cabellos('indicios.csv'))

**b.** Debería devolver el promedio de una lista de profundidades, pero da error:

In [ ]:
def promedio_profundidad(valores):
    # suma y cant nunca se inicializaban y cant sumaba d en vez de contar de a uno:
    # numpy ya lo resuelve con .mean()
    return np.array(valores).mean()

print(promedio_profundidad([22, 24, 21, 23, 25]))  # 23.0